# Module 0: Quantum Mechanics Fundamentals & Matrix Mechanics
## Notebook 4: Generalized Measurements, POVMs & Neumark's Dilation

---

### Executive Overview & Theoretical Foundations

In introductory quantum mechanics, measurement is typically introduced via the **von Neumann projection postulate**: an observable $\hat{A} = \sum a_m \hat{P}_m$ is associated with orthogonal projection operators $\hat{P}_m = |m\rangle\langle m|$ satisfying $\hat{P}_m \hat{P}_{m'} = \delta_{mm'} \hat{P}_m$ and $\sum_m \hat{P}_m = \hat{I}$. 

However, in quantum optics and quantum information, projective measurements are severely restrictive:
1. **Non-Orthogonal States Cannot Be Reliably Distinguished**: No projective measurement can unambiguously distinguish non-orthogonal states $|\psi_1\rangle$ and $|\psi_2\rangle$ ($\langle\psi_1|\psi_2\rangle \ne 0$) without introducing fatal error.
2. **Measurement with Ancillary Degrees of Freedom**: In realistic optical detection (e.g. balanced homodyne, heterodyne, or avalanche photodiodes), the signal field interacts with an auxiliary probe (local oscillator or detector atoms) before a macroscopic click occurs.

To capture the most general quantum measurements allowed by quantum theory, we formulate **Positive Operator-Valued Measures (POVMs)** and **Neumark's Dilation Theorem**.

---

#### 1. The POVM Formalism
A **Positive Operator-Valued Measure (POVM)** is a collection of Hermitian operators $\{ \hat{E}_m \}$ acting on Hilbert space $\mathcal{H}$ such that:
1. **Positivity**: $\hat{E}_m \ge 0$ for all $m$ (all eigenvalues are non-negative, $\langle\psi|\hat{E}_m|\psi\rangle \ge 0$).
2. **Completeness (Partition of Unity)**:
   $$\sum_{m} \hat{E}_m = \hat{I}$$

For a quantum state described by density operator $\hat{\rho}$, the probability of obtaining measurement outcome $m$ is given by the generalized Born rule:
$$p(m) = \mathrm{Tr}\left( \hat{E}_m \hat{\rho} \right)$$
Because $\sum_m \hat{E}_m = \hat{I}$, probability conservation is guaranteed:
$$\sum_m p(m) = \mathrm{Tr}\left( \sum_m \hat{E}_m \hat{\rho} \right) = \mathrm{Tr}(\hat{\rho}) = 1$$

##### Kraus Decomposition & State Update
Each POVM element can be factored into Kraus measurement operators $\hat{M}_m$:
$$\hat{E}_m = \hat{M}_m^\dagger \hat{M}_m$$
Upon obtaining outcome $m$, the post-measurement quantum state updates according to the generalized Lüders rule:
$$\hat{\rho} \longrightarrow \hat{\rho}_m = \frac{\hat{M}_m \hat{\rho} \hat{M}_m^\dagger}{\mathrm{Tr}(\hat{M}_m^\dagger \hat{M}_m \hat{\rho})}$$

---

#### 2. Unambiguous State Discrimination (USD)
Consider two non-orthogonal states with prior probabilities $\eta_1 = \eta_2 = 1/2$:
$$|\psi_1\rangle = \cos\theta |0\rangle + \sin\theta |1\rangle, \quad |\psi_2\rangle = \cos\theta |0\rangle - \sin\theta |1\rangle$$
with overlap $\langle\psi_1|\psi_2\rangle = \cos(2\theta) \ne 0$.

A projective measurement cannot distinguish them without errors. However, a 3-outcome POVM $\{\hat{E}_1, \hat{E}_2, \hat{E}_0\}$ can perform **error-free discrimination**:
- $\hat{E}_1 \propto |\psi_2^\perp\rangle\langle\psi_2^\perp|$: if outcome 1 clicks, the state is *guaranteed* to be $|\psi_1\rangle$ ($\mathrm{Tr}(\hat{E}_1 |\psi_2\rangle\langle\psi_2|) = 0$).
- $\hat{E}_2 \propto |\psi_1^\perp\rangle\langle\psi_1^\perp|$: if outcome 2 clicks, the state is *guaranteed* to be $|\psi_2\rangle$ ($\mathrm{Tr}(\hat{E}_2 |\psi_1\rangle\langle\psi_1|) = 0$).
- $\hat{E}_0 = \hat{I} - \hat{E}_1 - \hat{E}_2$: represents an **inconclusive outcome** ("I don't know"), which occurs with probability:
  $$P_{\text{inconclusive}} = |\langle\psi_1|\psi_2\rangle| = \cos(2\theta)$$
The maximum unambiguous success probability is the Ivanovic-Dieks-Peres (IDP) limit:
$$P_{\text{USD}} = 1 - |\langle\psi_1|\psi_2\rangle|$$

---

#### 3. Neumark's Dilation Theorem
**Theorem**: *Any POVM measurement $\{\hat{E}_m\}$ on a system $\mathcal{H}_S$ can be realized as a standard von Neumann projective measurement on an enlarged Hilbert space $\mathcal{H}_S \otimes \mathcal{H}_A$ combining the system with an ancillary environment.*

Mathematically, there exists an ancilla state $|0\rangle_A$, a unitary interaction $\hat{U}$, and orthogonal projectors $\hat{\Pi}_m = \hat{I}_S \otimes |m\rangle_A\langle m|_A$ such that:
$$p(m) = \mathrm{Tr}_S\left( \hat{E}_m \hat{\rho}_S \right) = \mathrm{Tr}_{SA}\left( \hat{\Pi}_m \hat{U} (\hat{\rho}_S \otimes |0\rangle_A\langle 0|_A) \hat{U}^\dagger \right)$$
This theorem proves that POVMs are not merely mathematical curiosities; they represent the exact physics of an open system coupled to an apparatus!

---

### Notebook Implementation Roadmap
1. **POVM Engine (`POVMMeasurementEngine`)**: Object-oriented suite for POVM verification, Kraus operators, Trine POVMs, and USD solvers.
2. **Neumark Dilation Constructor**: Explicit matrix dilation embedding arbitrary POVMs into unitary rotations on $\mathcal{H}_S \otimes \mathcal{H}_A$.
3. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: Bloch Representation of Trine POVM elements.
   - *Panel (b)*: USD Success Probability vs. Overlap compared against Helstrom bound.
   - *Panel (c)*: Dilation Unitary Matrix Sparsity and Coupling.
   - *Panel (d)*: Monte Carlo Stochastic Realization vs. Analytical Born Probabilities.
4. **Automated Verification Suite**: 5 unit tests validating completeness, positivity, zero-error USD, and Neumark projection.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

print(f"NumPy version: {np.__version__}")
print("Generalized Measurements & POVM environment initialized successfully.")


### Algorithmic Architecture: The POVMMeasurementEngine

The `POVMMeasurementEngine` provides rigorous tools for generalized quantum measurements:
1. **Completeness & Positivity Auditing**: Validates $\|\sum E_m - I\| < 10^{-12}$ and eigenvalues $\lambda_i(E_m) \ge -\epsilon$.
2. **The Trine POVM**: Symmetric 3-element POVM on a single qubit separated by $120^\circ$ on the Bloch sphere:
   $$\hat{E}_k = \frac{2}{3} |\phi_k\rangle\langle\phi_k|, \quad |\phi_k\rangle = \cos\left(\frac{k\pi}{3}\right)|0\rangle + \sin\left(\frac{k\pi}{3}\right)|1\rangle, \quad k \in \{0, 1, 2\}$$
3. **Optimal USD Solver**: Computes error-free discrimination operators for arbitrary non-orthogonal state pairs.
4. **Neumark Dilation Generator**: Constructs the exact unitary $U$ mapping $|\psi\rangle_S \otimes |0\rangle_A \to \sum_m \sqrt{\hat{E}_m}|\psi\rangle_S \otimes |m\rangle_A$.


In [ ]:
# Cell 4: Complete Object-Oriented Engine Implementation: POVMMeasurementEngine

class POVMMeasurementEngine:
    '''
    Comprehensive computational suite for Positive Operator-Valued Measures (POVMs),
    Unambiguous State Discrimination (USD), and Neumark's Dilation Theorem.
    '''
    
    @staticmethod
    def verify_povm(povm_elements: List[np.ndarray], tol: float = 1e-12) -> Tuple[bool, float, float]:
        '''
        Checks if a set of matrices form a valid POVM:
        1. Completeness: sum(E_m) == I
        2. Positivity: all eigenvalues of E_m >= 0
        '''
        d = povm_elements[0].shape[0]
        completeness_sum = np.zeros((d, d), dtype=np.complex128)
        min_eigenvalue = float('inf')
        
        for E in povm_elements:
            completeness_sum += E
            evals = la.eigvalsh(0.5 * (E + E.T.conj()))
            min_eigenvalue = min(min_eigenvalue, float(np.min(evals)))
            
        completeness_error = float(la.norm(completeness_sum - np.eye(d)))
        is_valid = (completeness_error < tol) and (min_eigenvalue >= -tol)
        return is_valid, completeness_error, min_eigenvalue

    @classmethod
    def trine_povm(cls) -> List[np.ndarray]:
        '''
        Constructs the symmetric 3-element Trine POVM on a single qubit (120 degrees on Bloch equator).
        E_k = (2/3) |phi_k><phi_k| for k = 0, 1, 2.
        '''
        angles = [0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0]
        elements = []
        for phi in angles:
            # State vector in equatorial plane: (|0> + exp(i*phi)|1>) / sqrt(2)
            state = np.array([1.0, np.exp(1.0j * phi)], dtype=np.complex128) / np.sqrt(2.0)
            E_k = (2.0 / 3.0) * np.outer(state, state.conj())
            elements.append(E_k)
        return elements

    @classmethod
    def unambiguous_state_discrimination_povm(
        cls, 
        psi_1: np.ndarray, 
        psi_2: np.ndarray
    ) -> Tuple[List[np.ndarray], float]:
        '''
        Constructs 3-element POVM {E_1, E_2, E_0} for Unambiguous State Discrimination of psi_1 and psi_2.
        E_1 detects psi_1 (zero overlap with psi_2)
        E_2 detects psi_2 (zero overlap with psi_1)
        E_0 is the inconclusive operator.
        '''
        overlap = abs(np.vdot(psi_1, psi_2))
        
        # Orthogonal complements
        # psi_2_perp is perpendicular to psi_2
        psi_2_perp = np.array([-psi_2[1].conj(), psi_2[0].conj()], dtype=np.complex128)
        psi_2_perp /= la.norm(psi_2_perp)
        
        # psi_1_perp is perpendicular to psi_1
        psi_1_perp = np.array([-psi_1[1].conj(), psi_1[0].conj()], dtype=np.complex128)
        psi_1_perp /= la.norm(psi_1_perp)
        
        # Scaling factor for optimal USD with equal priors: s = 1 / (1 + |<1|2>|)
        scale = 1.0 / (1.0 + overlap)
        E_1 = scale * np.outer(psi_2_perp, psi_2_perp.conj())
        E_2 = scale * np.outer(psi_1_perp, psi_1_perp.conj())
        E_0 = np.eye(2, dtype=np.complex128) - E_1 - E_2
        
        success_prob = 1.0 - overlap
        return [E_1, E_2, E_0], success_prob

    @classmethod
    def neumark_dilation_unitary(cls, povm_elements: List[np.ndarray]) -> np.ndarray:
        '''
        Constructs Neumark dilation unitary U acting on H_S (dim d) (x) H_A (dim M).
        First column block maps |psi>_S (x) |0>_A to sum_m M_m |psi>_S (x) |m>_A.
        '''
        d = povm_elements[0].shape[0]
        M = len(povm_elements)
        dim_total = d * M
        
        # Compute Kraus operators M_m = sqrt(E_m)
        Kraus = []
        for E in povm_elements:
            evals, evecs = la.eigh(0.5 * (E + E.T.conj()))
            sqrt_evals = np.sqrt(np.maximum(0.0, evals))
            M_m = evecs @ np.diag(sqrt_evals) @ evecs.T.conj()
            Kraus.append(M_m)
            
        # First column block of size (d*M, d)
        V = np.zeros((dim_total, d), dtype=np.complex128)
        for m in range(M):
            V[m * d : (m + 1) * d, :] = Kraus[m]
            
        # Complete V into an orthogonal unitary matrix via Gram-Schmidt / QR
        q, _ = la.qr(V, mode='full')
        return q

    @staticmethod
    def sample_povm_outcomes(
        rho: np.ndarray, 
        povm_elements: List[np.ndarray], 
        n_shots: int = 2000
    ) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Simulates Monte Carlo sampling of POVM measurement outcomes.
        Returns analytical probabilities and sampled empirical counts.
        '''
        probs = np.array([float(np.real(np.trace(rho @ E))) for E in povm_elements])
        probs = np.maximum(0.0, probs)
        probs /= np.sum(probs)
        
        counts = np.random.multinomial(n_shots, probs)
        return probs, counts

print("POVMMeasurementEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Grade 4-Panel Visualization Suite

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Generalized Quantum Measurements: POVMs & Neumark's Dilation", fontsize=16, fontweight="bold", y=0.98)

# ------------------------------------------------------------------
# Panel (a): Bloch Representation of Trine POVM
# ------------------------------------------------------------------
ax_a = axes[0, 0]
trine_elements = POVMMeasurementEngine.trine_povm()
theta_circle = np.linspace(0, 2*np.pi, 200)
ax_a.plot(np.cos(theta_circle), np.sin(theta_circle), color="gray", linestyle="--", lw=1.2, label="Bloch Equator")

angles = [0.0, 2.0*np.pi/3.0, 4.0*np.pi/3.0]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for idx, (ang, col) in enumerate(zip(angles, colors)):
    rx = np.cos(ang)
    ry = np.sin(ang)
    ax_a.arrow(0, 0, rx*0.9, ry*0.9, head_width=0.08, head_length=0.08, fc=col, ec=col, lw=2.2)
    ax_a.text(rx*1.15, ry*1.15, rf"$\hat{{E}}_{idx}$ ($120^\circ$)", ha="center", va="center", fontsize=11, fontweight="bold", color=col)

ax_a.scatter([0], [0], color="black", s=40, zorder=5)
ax_a.set_title(r"(a) Trine POVM Vectors on the Bloch Sphere ($|\vec{r}_k|=2/3$)", fontsize=13, fontweight="bold")
ax_a.set_xlabel(r"Bloch $r_x$")
ax_a.set_ylabel(r"Bloch $r_y$")
ax_a.set_xlim(-1.4, 1.4)
ax_a.set_ylim(-1.4, 1.4)
ax_a.set_aspect("equal")
ax_a.grid(True, linestyle=":", alpha=0.6)
ax_a.legend(loc="upper right", framealpha=0.9)

# ------------------------------------------------------------------
# Panel (b): Unambiguous State Discrimination vs. Helstrom Bound
# ------------------------------------------------------------------
ax_b = axes[0, 1]
theta_scan = np.linspace(0.01, np.pi/4.0 - 0.01, 100)
usd_success = []
helstrom_bounds = []

for th in theta_scan:
    p1 = np.array([np.cos(th), np.sin(th)], dtype=np.complex128)
    p2 = np.array([np.cos(th), -np.sin(th)], dtype=np.complex128)
    _, p_usd = POVMMeasurementEngine.unambiguous_state_discrimination_povm(p1, p2)
    usd_success.append(p_usd)
    # Helstrom bound for minimum error: P_opt = 0.5 * (1 + sqrt(1 - |<1|2>|^2))
    overlap = abs(np.vdot(p1, p2))
    p_helstrom = 0.5 * (1.0 + np.sqrt(1.0 - overlap**2))
    helstrom_bounds.append(p_helstrom)

overlap_scan = np.cos(2.0 * theta_scan)
ax_b.plot(overlap_scan, usd_success, color="#d62728", lw=2.5, label=r"Unambiguous (Error-Free) $P_{\mathrm{USD}} = 1 - |\langle\psi_1|\psi_2\rangle|$")
ax_b.plot(overlap_scan, helstrom_bounds, color="#1f77b4", lw=2.2, linestyle="--", label=r"Helstrom Bound (With Errors) $P_{\mathrm{Helstrom}}$")
ax_b.fill_between(overlap_scan, usd_success, color="#d62728", alpha=0.15)

ax_b.set_title(r"(b) State Discrimination Limits vs. Overlap $|\langle\psi_1|\psi_2\rangle|$", fontsize=13, fontweight="bold")
ax_b.set_xlabel(r"State Overlap $|\langle\psi_1|\psi_2\rangle|$")
ax_b.set_ylabel(r"Success Probability")
ax_b.set_xlim(0.0, 1.0)
ax_b.set_ylim(0.0, 1.05)
ax_b.grid(True, linestyle=":", alpha=0.6)
ax_b.legend(loc="upper right", framealpha=0.9)

# ------------------------------------------------------------------
# Panel (c): Neumark Dilation Unitary Matrix Sparsity
# ------------------------------------------------------------------
ax_c = axes[1, 0]
U_dilation = POVMMeasurementEngine.neumark_dilation_unitary(trine_elements)
im_c = ax_c.imshow(np.abs(U_dilation), cmap="Blues", origin="upper")
cbar_c = fig.colorbar(im_c, ax=ax_c, fraction=0.046, pad=0.04)
cbar_c.set_label(r"Matrix Element Magnitude $|U_{ij}|$", fontsize=11)

ax_c.set_title(r"(c) Neumark Dilation Unitary Structure on $\mathcal{H}_S \otimes \mathcal{H}_A$ ($6\times 6$)", fontsize=13, fontweight="bold")
ax_c.set_xlabel(r"Total Basis Column Index")
ax_c.set_ylabel(r"Total Basis Row Index")

# ------------------------------------------------------------------
# Panel (d): Monte Carlo POVM Sampling vs. Analytical Born Probabilities
# ------------------------------------------------------------------
ax_d = axes[1, 1]
psi_test = np.array([np.cos(np.pi/6), np.sin(np.pi/6)], dtype=np.complex128)
rho_test = np.outer(psi_test, psi_test.conj())
n_shots = 3000
ana_probs, counts = POVMMeasurementEngine.sample_povm_outcomes(rho_test, trine_elements, n_shots=n_shots)
emp_probs = counts / n_shots

x_indices = np.arange(3)
width = 0.35
ax_d.bar(x_indices - width/2, ana_probs, width, color="#1f77b4", alpha=0.85, label=r"Born Rule $\mathrm{Tr}(\hat{E}_m \hat{\rho})$")
ax_d.bar(x_indices + width/2, emp_probs, width, color="#ff7f0e", alpha=0.85, label=rf"Empirical ($N = {n_shots}$ shots)")

for i in range(3):
    ax_d.text(x_indices[i] - width/2, ana_probs[i] + 0.015, f"{ana_probs[i]:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax_d.text(x_indices[i] + width/2, emp_probs[i] + 0.015, f"{emp_probs[i]:.3f}", ha="center", color="#d62728", fontsize=9, fontweight="bold")

ax_d.set_title(r"(d) Empirical POVM Sampling vs. Analytical Born Probabilities", fontsize=13, fontweight="bold")
ax_d.set_xticks(x_indices)
ax_d.set_xticklabels([r"Outcome $\hat{E}_0$", r"Outcome $\hat{E}_1$", r"Outcome $\hat{E}_2$"])
ax_d.set_ylabel(r"Probability")
ax_d.set_ylim(0.0, 0.65)
ax_d.grid(True, linestyle=":", alpha=0.6)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()


### In-Depth Physical Interpretation & Experimental Frontier

#### 1. Why Projective Measurements Are Insufficient
In classical detection, determining whether a light pulse exists does not disturb the surrounding universe. In quantum optics, a standard von Neumann projection collapses a state into an orthogonal subspace. If two laser states $|\psi_1\rangle$ and $|\psi_2\rangle$ are non-orthogonal, any projective measurement has a nonzero minimum error given by the **Helstrom bound**:
$$P_{\text{error}}^{\text{Helstrom}} = \frac{1}{2}\left(1 - \sqrt{1 - |\langle\psi_1|\psi_2\rangle|^2}\right)$$
In contrast, **Unambiguous State Discrimination (USD)** via POVMs guarantees that whenever an outcome clicks, the identification is **100% error-free**, offloading the quantum uncertainty into a third, inconclusive detector outcome $\hat{E}_0$. This is the operational engine behind B92 quantum cryptography!

#### 2. Experimental Realization of Neumark's Dilation
Neumark's dilation proves that any POVM can be implemented by:
1. Coupling the incoming photon mode with an ancillary vacuum mode using a **balanced beam splitter** (which enacts the unitary $\hat{U}$).
2. Performing standard single-photon counting (projective measurements) on the output ports.
Thus, **multi-port optical networks are physical physical embodiments of Neumark dilation**!


In [ ]:
# Cell 7: Interactive Exploration & POVM Diagnostic Suite

def analyze_povm_measurement_system(
    theta_usd: float = np.pi / 6.0, 
    n_shots: int = 5000
) -> Dict[str, Union[float, bool]]:
    '''
    Evaluates Trine and USD POVM characteristics, completeness, and sampling statistics.
    '''
    # 1. Trine POVM Audit
    trine = POVMMeasurementEngine.trine_povm()
    is_trine_valid, trine_err, min_eig = POVMMeasurementEngine.verify_povm(trine)
    
    # 2. USD POVM Audit
    psi_1 = np.array([np.cos(theta_usd), np.sin(theta_usd)], dtype=np.complex128)
    psi_2 = np.array([np.cos(theta_usd), -np.sin(theta_usd)], dtype=np.complex128)
    usd_elements, p_success = POVMMeasurementEngine.unambiguous_state_discrimination_povm(psi_1, psi_2)
    is_usd_valid, usd_err, _ = POVMMeasurementEngine.verify_povm(usd_elements)
    
    # Check zero error conditions
    err_1_on_2 = float(np.real(np.vdot(psi_2, usd_elements[0] @ psi_2)))
    err_2_on_1 = float(np.real(np.vdot(psi_1, usd_elements[1] @ psi_1)))
    
    # Neumark dilation unitarity check
    U_dil = POVMMeasurementEngine.neumark_dilation_unitary(trine)
    u_err = float(la.norm(U_dil.T.conj() @ U_dil - np.eye(U_dil.shape[0])))
    
    print("=" * 68)
    print("GENERALIZED QUANTUM MEASUREMENT (POVM) DIAGNOSTIC REPORT")
    print("=" * 68)
    print(f"Trine POVM Completeness Error:     ||sum E - I|| = {trine_err:.2e} (Valid: {is_trine_valid})")
    print(f"Trine Minimum Eigenvalue:          lambda_min    = {min_eig:+.2e} (Non-negative)")
    print("-" * 68)
    print(f"USD State Overlap:                 |<psi_1|psi_2>| = {abs(np.vdot(psi_1, psi_2)):.4f}")
    print(f"USD Theoretical Success Prob:      P_USD           = {p_success * 100:.2f}%")
    print(f"USD Error Rate (False Alarm 1):    Tr(E_1 * rho_2) = {err_1_on_2:.2e} (Target: 0.00)")
    print(f"USD Error Rate (False Alarm 2):    Tr(E_2 * rho_1) = {err_2_on_1:.2e} (Target: 0.00)")
    print("-" * 68)
    print(f"Neumark Dilation Unitarity Error:  ||U^dag U - I|| = {u_err:.2e} (Strictly Unitary)")
    print("=" * 68)
    
    return {
        "is_trine_valid": is_trine_valid,
        "is_usd_valid": is_usd_valid,
        "p_success": p_success,
        "err_1_on_2": err_1_on_2,
        "err_2_on_1": err_2_on_1,
        "u_err": u_err
    }

diag_results = analyze_povm_measurement_system(theta_usd=np.pi/6.0, n_shots=5000)


### Rigorous Automated Verification & Acceptance Suite

The generalized measurement engine is subjected to 5 automated unit tests:
1. **Test 1: Trine POVM Completeness & Positivity**: Validates $\sum E_m = \hat{I}$ and $E_m \ge 0$ to within $10^{-14}$.
2. **Test 2: Zero-Error Guarantee in USD**: Asserts that $\mathrm{Tr}(\hat{E}_1 |\psi_2\rangle\langle\psi_2|) \equiv 0$ and $\mathrm{Tr}(\hat{E}_2 |\psi_1\rangle\langle\psi_1|) \equiv 0$.
3. **Test 3: IDP Success Probability Bound**: Confirms $P_{\text{USD}} = 1 - |\langle\psi_1|\psi_2\rangle|$ exactly.
4. **Test 4: Neumark Dilation Unitarity**: Verifies that the dilated coupling operator is strictly unitary $\|U^\dagger U - I\| < 10^{-14}$.
5. **Test 5: Probability Conservation**: Confirms that for any arbitrary mixed state $\hat{\rho}$, $\sum p(m) \equiv 1.00000000$.


In [ ]:
# Cell 9: Automated Benchmark & Acceptance Suite Implementation

def run_benchmark_suite():
    '''
    Executes automated unit tests verifying mathematical and physical accuracy of POVM simulations.
    '''
    print("=" * 70)
    print("STARTING GENERALIZED MEASUREMENTS (POVM) BENCHMARK SUITE")
    print("=" * 70)
    
    # ------------------------------------------------------------------
    # Test 1: Trine POVM Completeness & Positivity
    # ------------------------------------------------------------------
    print("Test 1: Verifying Trine POVM Completeness and Positivity...")
    trine = POVMMeasurementEngine.trine_povm()
    is_valid, comp_err, min_eig = POVMMeasurementEngine.verify_povm(trine)
    print(f"  -> Completeness error: {comp_err:.2e} (Threshold: < 1e-12)")
    print(f"  -> Minimum eigenvalue: {min_eig:+.2e} (Threshold: >= -1e-12)")
    assert is_valid, f"Trine POVM is not valid: comp_err={comp_err}, min_eig={min_eig}"
    print("  [PASS] Trine POVM strictly satisfies partition of unity and positivity.")

    # ------------------------------------------------------------------
    # Test 2: Zero-Error Condition in USD
    # ------------------------------------------------------------------
    print("\nTest 2: Verifying Zero-Error Guarantee in Unambiguous State Discrimination...")
    theta = 0.35
    psi_1 = np.array([np.cos(theta), np.sin(theta)], dtype=np.complex128)
    psi_2 = np.array([np.cos(theta), -np.sin(theta)], dtype=np.complex128)
    usd_elements, _ = POVMMeasurementEngine.unambiguous_state_discrimination_povm(psi_1, psi_2)
    
    err_1_on_2 = abs(np.vdot(psi_2, usd_elements[0] @ psi_2))
    err_2_on_1 = abs(np.vdot(psi_1, usd_elements[1] @ psi_1))
    print(f"  -> Error Tr(E_1 * rho_2): {err_1_on_2:.2e} (Target: 0.00)")
    print(f"  -> Error Tr(E_2 * rho_1): {err_2_on_1:.2e} (Target: 0.00)")
    assert err_1_on_2 < 1e-14, f"USD error 1 on 2 failed: {err_1_on_2}"
    assert err_2_on_1 < 1e-14, f"USD error 2 on 1 failed: {err_2_on_1}"
    print("  [PASS] USD operators produce strictly zero false-alarm errors.")

    # ------------------------------------------------------------------
    # Test 3: IDP Success Probability Limit
    # ------------------------------------------------------------------
    print("\nTest 3: Verifying Ivanovic-Dieks-Peres (IDP) Success Bound...")
    overlap = abs(np.vdot(psi_1, psi_2))
    expected_p = 1.0 - overlap
    _, actual_p = POVMMeasurementEngine.unambiguous_state_discrimination_povm(psi_1, psi_2)
    print(f"  -> Computed success probability: {actual_p:.6f}")
    print(f"  -> Theoretical IDP bound:       {expected_p:.6f}")
    assert abs(actual_p - expected_p) < 1e-12, f"IDP bound mismatch: {actual_p} vs {expected_p}"
    print("  [PASS] USD success probability matches analytical IDP bound.")

    # ------------------------------------------------------------------
    # Test 4: Neumark Dilation Unitarity
    # ------------------------------------------------------------------
    print("\nTest 4: Verifying Neumark Dilation Operator Unitarity ||U^dag U - I||...")
    U_dil = POVMMeasurementEngine.neumark_dilation_unitary(trine)
    u_err = la.norm(U_dil.T.conj() @ U_dil - np.eye(U_dil.shape[0]))
    print(f"  -> Unitarity error: {u_err:.2e} (Threshold: < 1e-12)")
    assert u_err < 1e-12, f"Dilation operator is not unitary: {u_err}"
    print("  [PASS] Neumark dilation embeds POVM into an exact unitary interaction.")

    # ------------------------------------------------------------------
    # Test 5: Probability Conservation under Generalized Born Rule
    # ------------------------------------------------------------------
    print("\nTest 5: Verifying Probability Conservation for Random Mixed States...")
    # Generate random mixed state
    H_rand = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)
    rho_rand = H_rand @ H_rand.T.conj()
    rho_rand /= np.trace(rho_rand)
    
    probs = [float(np.real(np.trace(rho_rand @ E))) for E in trine]
    prob_sum = sum(probs)
    print(f"  -> Total Born probability sum: {prob_sum:.12f} (Target: 1.000000000000)")
    assert abs(prob_sum - 1.0) < 1e-12, f"Probability sum not unity: {prob_sum}"
    print("  [PASS] Generalized Born rule strictly satisfies probability conservation.")

    print("=" * 70)
    print("ALL 5 POVM BENCHMARK TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 70)

run_benchmark_suite()


### Summary of Key Results & Master Mathematical Formulary

| Concept / Quantity | Mathematical Expression | Physical Significance |
| :--- | :--- | :--- |
| **POVM Completeness** | $\sum_m \hat{E}_m = \hat{I}, \quad \hat{E}_m \ge 0$ | Generalized partition of unity guaranteeing $\sum p(m) = 1$ |
| **Generalized Born Rule** | $p(m) = \mathrm{Tr}\left(\hat{E}_m \hat{\rho}\right)$ | Probability of outcome $m$ for arbitrary quantum state $\hat{\rho}$ |
| **Kraus Measurement Operators** | $\hat{E}_m = \hat{M}_m^\dagger \hat{M}_m, \quad \hat{\rho} \to \frac{\hat{M}_m \hat{\rho}\hat{M}_m^\dagger}{\mathrm{Tr}(\hat{M}_m^\dagger \hat{M}_m \hat{\rho})}$ | State update rule describing quantum measurement backaction |
| **Trine POVM** | $\hat{E}_k = \frac{2}{3}|\phi_k\rangle\langle\phi_k|, \quad \phi_k = \frac{2\pi k}{3}$ | Symmetric 3-element POVM in 2D space separated by $120^\circ$ |
| **USD Success Probability** | $P_{\text{USD}} = 1 - |\langle\psi_1|\psi_2\rangle|$ | Maximum error-free discrimination probability between non-orthogonal states |
| **Neumark's Dilation** | $p(m) = \mathrm{Tr}_{SA}\left( (\hat{I} \otimes |m\rangle\langle m|) \hat{U} (\hat{\rho}_S \otimes |0\rangle\langle 0|) \hat{U}^\dagger \right)$ | Dilation of any POVM into unitary evolution on an enlarged ancilla space |
